In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/application_train.csv')
print(f"Original shape: {df.shape}")

Original shape: (307511, 122)


In [17]:
# Calculate missing percentage
missing_pct = (df.isnull().sum() / len(df)) * 100

# Protect high-value features even if they have >40% missing
protected_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

# Drop columns with >40% missing EXCEPT protected ones
cols_to_drop = [
    col for col in missing_pct[missing_pct > 40].index.tolist()
    if col not in protected_cols
]

print(f"Columns to drop (>40% missing): {len(cols_to_drop)}")
print(cols_to_drop)

# Drop them
df.drop(columns=cols_to_drop, inplace=True)
print(f"Shape after dropping: {df.shape}")

Columns to drop (>40% missing): 48
['OWN_CAR_AGE', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE', 'WALLSMATERIAL_MODE', '

In [18]:
# 365243 is a placeholder for unemployed — replace with NaN
print(f"Before fix: {(df['DAYS_EMPLOYED'] == 365243).sum()} anomalous rows")

df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

print(f"After fix: {(df['DAYS_EMPLOYED'] == 365243).sum()} anomalous rows")

Before fix: 55374 anomalous rows
After fix: 0 anomalous rows


In [19]:
# DAYS_BIRTH and DAYS_EMPLOYED are negative (days relative to application date)
# Convert to positive and meaningful units

df['AGE_YEARS'] = np.abs(df['DAYS_BIRTH']) / 365
df['YEARS_EMPLOYED'] = np.abs(df['DAYS_EMPLOYED']) / 365

print("Sample AGE_YEARS:", df['AGE_YEARS'].describe())
print("\nSample YEARS_EMPLOYED:", df['YEARS_EMPLOYED'].describe())

Sample AGE_YEARS: count    307511.000000
mean         43.936973
std          11.956133
min          20.517808
25%          34.008219
50%          43.150685
75%          53.923288
max          69.120548
Name: AGE_YEARS, dtype: float64

Sample YEARS_EMPLOYED: count    252137.000000
mean          6.531971
std           6.406466
min           0.000000
25%           2.101370
50%           4.515068
75%           8.698630
max          49.073973
Name: YEARS_EMPLOYED, dtype: float64


In [20]:
# Check remaining missing
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f"Columns still missing: {len(remaining_missing)}")

# Separate numerical and categorical
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Impute numerical with median (robust to outliers)
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Impute categorical with mode
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Verify
print(f"Missing values remaining: {df.isnull().sum().sum()}")

Columns still missing: 21
Missing values remaining: 0


In [21]:
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"Duplicates removed: {before - after}")
print(f"Shape after dedup: {df.shape}")

Duplicates removed: 0
Shape after dedup: (307511, 76)


In [22]:
# Check categorical columns remaining
print("Categorical columns:")
print(df[cat_cols].nunique().sort_values())

Categorical columns:
NAME_CONTRACT_TYPE             2
FLAG_OWN_CAR                   2
FLAG_OWN_REALTY                2
CODE_GENDER                    3
NAME_EDUCATION_TYPE            5
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
NAME_TYPE_SUITE                7
WEEKDAY_APPR_PROCESS_START     7
NAME_INCOME_TYPE               8
OCCUPATION_TYPE               18
ORGANIZATION_TYPE             58
dtype: int64


In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Binary categoricals — Label Encode
binary_cats = [col for col in cat_cols if df[col].nunique() == 2]
print(f"Binary categoricals: {binary_cats}")

for col in binary_cats:
    df[col] = le.fit_transform(df[col].astype(str))

Binary categoricals: ['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']


In [24]:
# Multi-class categoricals — One Hot Encode
multi_cats = [col for col in cat_cols if df[col].nunique() > 2]
print(f"Multi-class categoricals: {multi_cats}")

df = pd.get_dummies(df, columns=multi_cats, drop_first=True)
print(f"Shape after encoding: {df.shape}")

Multi-class categoricals: ['CODE_GENDER', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']
Shape after encoding: (307511, 176)


In [25]:
# SK_ID_CURR is just a row identifier — no predictive value
df.drop(columns=['SK_ID_CURR'], inplace=True)
print(f"Final shape: {df.shape}")

Final shape: (307511, 175)


In [26]:
df.to_csv('../data/processed/cleaned_data.csv', index=False)
print("Cleaned data saved to data/processed/cleaned_data.csv")
print(f"Final dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

Cleaned data saved to data/processed/cleaned_data.csv
Final dataset: 307,511 rows, 175 columns


In [27]:
# Final checks
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
print("Target distribution:")
print(df['TARGET'].value_counts(normalize=True).round(3))

Missing values: 0
Duplicates: 0
Target distribution:
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64
